# 🤖 Mastering LLM Parameters: A Deep Dive into Zero-Shot Prompting

## Introduction

Welcome to this comprehensive exploration of Large Language Model (LLM) parameters using NVIDIA’s Llama-3.1 Nemotron Ultra model. In this notebook, you’ll learn:

- Core LLM parameters: **temperature**, **top_p**, **max_tokens**, **top_k**  
- How each parameter influences model output  
- Practical zero-shot prompting examples  
- Best practices and recommended settings  


## 🎬 Before We Begin: Why Tuning LLM Parameters Matters

When you send a prompt to a Large Language Model (LLM), you don’t just supply text—you also control **how** the model writes its response. By adjusting a handful of key parameters, you can tailor the output’s **creativity**, **focus**, **length**, and **consistency**. Below is a quick‐reference “cheat sheet” for the most common settings.

---

### 🌡️ Temperature  
- **What it does:** Controls randomness in token selection  
- **Range:** 0.0 (greedy/deterministic) → 1.0 (maximally random)  
- **When to use:**  
  - **Low (0–0.3):** factual answers, code generation, translations  
  - **Medium (0.4–0.7):** balanced explanations, summaries  
  - **High (0.8–1.0):** creative writing, brainstorming, storytelling  

---

### 🎯 Top-P (Nucleus Sampling)  
- **What it does:** Keeps only the smallest set of next-token candidates whose cumulative probability ≤ p  
- **Range:** 0.0 (almost greedy) → 1.0 (full vocabulary)  
- **When to use:**  
  - **Low (0.1–0.3):** very focused, avoids rare words  
  - **Medium (0.4–0.8):** balanced diversity  
  - **High (0.9–1.0):** broad exploration of ideas  

---

### 🔢 Top-K  
- **What it does:** Limits sampling to the K most likely next tokens  
- **Values:** 1 (greedy) → thousands (near full vocabulary)  
- **When to use:**  
  - **K = 1:** pure greedy decoding  
  - **K = 20–50:** good balance of quality and diversity  
  - **K > 100:** very diverse (but can get off-topic)  

---

### 📏 Max Tokens  
- **What it does:** Caps the total length of the generated response (in tokens)  
- **Tip:** 1 token ≈ 0.75 words  
- **When to use:**  
  - **50–100:** short answers, titles, tweets  
  - **200–500:** paragraphs, summaries  
  - **500+:** long‐form essays, detailed explanations  

---

### 🚀 Putting It All Together  
- **Conservative (e.g., factual QA):**  
  ```yaml
    temperature: 0.0  
    top_p: 0.1  
    max_tokens: 150

- **Balanced (general‐purpose):**  
  ```yaml
    temperature: 0.5  
    top_p: 0.7  
    max_tokens: 200  


- **Creative (storytelling):**  
  ```yaml
    temperature: 0.9  
    top_p: 0.95  
    max_tokens: 300
    

## 📚 1. Setting Up Our Environment

We’ll import necessary libraries and securely load our NVIDIA API key from Colab secrets.


In [ ]:
# Install dependencies (run once)
!pip install openai huggingface-hub

# Imports & API key
import os, time
from openai import OpenAI
from google.colab import userdata

# Load NVIDIA key from Colab secrets
api_key = userdata.get("NVDIA-KEY")
if not api_key:
    raise ValueError("Please add NVIDIA_API_KEY to Colab secrets.")
os.environ["NVIDIA_API_KEY"] = api_key

# Initialize OpenAI client pointed at NVIDIA endpoint
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)
print("✅ Environment ready, client initialized.")

✅ Environment ready, client initialized.


## 🏗️ 2. Building Our `ZeroShotLLM` Class

Encapsulate the API setup and provide a clean interface for generating responses with custom parameters.


In [ ]:
class ZeroShotLLM:
    def __init__(self, model: str = "nvidia/llama-3.1-nemotron-ultra-253b-v1"):
        """
        Initialize the LLM wrapper with NVIDIA API key and model name.
        """
        self.api_key = os.environ["NVIDIA_API_KEY"]
        self.client = OpenAI(base_url="https://integrate.api.nvidia.com/v1",
                              api_key=self.api_key)
        self.model = model
        print(f"✅ ZeroShotLLM initialized for model: {self.model}")

## 🎛️ 3. The `generate_response` Method

This method drives the zero-shot prompting. Key parameters:

- **temperature** (0.0–1.0): lower → deterministic, higher → creative  
- **top_p** (0.0–1.0): nucleus sampling threshold  
- **max_tokens**: maximum tokens in output  
- **top_k**: limit sampling to top K tokens (if supported)  


In [ ]:
def generate_response(self,
                      prompt: str,
                      temperature: float = 0.1,
                      max_tokens: int = 256,
                      top_p: float = 0.95,
                      top_k: int = None) -> str:
    """
    Send a zero-shot prompt and return the model’s response.
    """
    params = {
        "model": self.model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": top_p
    }
    if top_k is not None:
        params["top_k"] = top_k

    resp = self.client.chat.completions.create(**params)
    return resp.choices[0].message.content.strip()

# Attach to class
ZeroShotLLM.generate_response = generate_response
print("🔧 Added generate_response method to ZeroShotLLM.")

🔧 Added generate_response method to ZeroShotLLM.


## 🧪 4. Initialize & Test Our LLM

Instantiate and run a simple sanity check to confirm everything is wired correctly.


In [ ]:
llm = ZeroShotLLM()
prompt = "Explain the concept of artificial intelligence in simple terms."
response = llm.generate_response(prompt)
print("🤖 PROMPT:\n", prompt)
print("\n📝 RESPONSE:\n", response)


✅ ZeroShotLLM initialized for model: nvidia/llama-3.1-nemotron-ultra-253b-v1
🤖 PROMPT:
 Explain the concept of artificial intelligence in simple terms.

📝 RESPONSE:
 <think>
Okay, I need to explain artificial intelligence in simple terms. Let me start by recalling what I know. AI refers to machines that can perform tasks that typically require human intelligence. But how do I break that down without using jargon?

Maybe start with examples. People use AI every day, like Siri or Google Maps. But wait, those are applications of AI. The core idea is that machines can learn and make decisions on their own. But how is that different from regular computers?

Regular computers follow programmed instructions exactly. AI, on the other hand, can adapt and improve over time. So, it's about learning from experience. Like how a child learns from mistakes, AI systems learn from data.

Wait, machine learning is a part of AI. So, AI is the broader concept, and machine learning is a method used to achi

## 🌡️ 5. Experiment 1: Temperature Effect

Test how **temperature** influences creativity vs. consistency:

- **0.0**: Fully deterministic  
- **0.5**: Balanced  
- **1.0**: Highly creative  


In [ ]:
def test_temperature(llm, prompt):
    print("🌡️ Temperature Effect Test")
    for temp in [0.0, 0.5, 1.0]:
        out = llm.generate_response(prompt, temperature=temp, max_tokens=100)
        print(f"\n🔹 temperature={temp}\n{out[:200]}…")

test_temperature(llm, "Write a short story about a robot learning to paint.")


🌡️ Temperature Effect Test

🔹 temperature=0.0
<think>
Okay, the user wants a short story about a robot learning to paint. Let me start by thinking about the main elements. The robot should have a journey from not understanding art to creating som…

🔹 temperature=0.5
<think>
Okay, the user wants a short story about a robot learning to paint. Let me start by brainstorming some ideas. So, the main character is a robot. Maybe I can name it something techy but relatab…

🔹 temperature=1.0
<think>
Okay, the user wants a short story about a robot learning to paint. Let me start by thinking about the main elements here. The robot's journey from not understanding art to creating something …


## 📏 6. Experiment 2: Max Tokens

Control **response length** by varying `max_tokens`:

- **50** tokens ≈ short answer  
- **200** tokens ≈ paragraph  
- **500** tokens ≈ longer explanation  


In [ ]:
def test_max_tokens(llm, prompt):
    print("📏 Max Tokens Effect Test")
    for tok in [50, 200, 500]:
        out = llm.generate_response(prompt, max_tokens=tok)
        words = len(out.split())
        print(f"\n🔹 max_tokens={tok} → {words} words\n{out[:200]}…")

test_max_tokens(llm, "Describe the potential applications of quantum computing.")


📏 Max Tokens Effect Test

🔹 max_tokens=50 → 42 words
<think>
Okay, so I need to describe the potential applications of quantum computing. Let me start by recalling what I know about quantum computing. From what I remember, quantum computers use qubits i…

🔹 max_tokens=200 → 166 words
<think>
Okay, so I need to describe the potential applications of quantum computing. Let me start by recalling what I know about quantum computing. From what I remember, quantum computers use qubits i…

🔹 max_tokens=500 → 416 words
<think>
Okay, so I need to describe the potential applications of quantum computing. Let me start by recalling what I know about quantum computing. From what I remember, quantum computers use qubits i…


## 🎯 7. Experiment 3: Top-P (Nucleus Sampling)

Adjust **top_p** to control token diversity:

- **0.1**: Very focused  
- **0.5**: Balanced  
- **0.95**: Broad sampling  


In [ ]:
def test_top_p(llm, prompt):
    print("🎯 Top-P Effect Test")
    for p in [0.1, 0.5, 0.95]:
        out = llm.generate_response(prompt, top_p=p, temperature=0.7)
        print(f"\n🔹 top_p={p}\n{out[:200]}…")

test_top_p(llm, "Describe a futuristic city in the year 2100.")


🎯 Top-P Effect Test

🔹 top_p=0.1
<think>
Okay, the user wants me to describe a futuristic city in the year 2100. Let me start by thinking about the key elements that would make a city futuristic. First, technology would have advanced…

🔹 top_p=0.5
<think>
Okay, the user wants me to describe a futuristic city in the year 2100. Let me start by thinking about the key elements that would make a city futuristic. First, technology would have advanced…

🔹 top_p=0.95
<think>
Okay, the user wants me to describe a futuristic city in 2100. Let me start by thinking about the key elements that would make a city futuristic. First, sustainability is a big concern now, so…


## ⚖️ 8. Parameter Combinations

Compare three profiles:

1. **Conservative**: `temperature=0.0, top_p=0.1`  
2. **Balanced**: `temperature=0.5, top_p=0.7`  
3. **Creative**: `temperature=0.9, top_p=0.95`  


In [ ]:
def compare_profiles(llm, prompt):
    configs = [
        ("🔒 Conservative", {"temperature":0.0, "top_p":0.1}),
        ("⚖️ Balanced",    {"temperature":0.5, "top_p":0.7}),
        ("🎨 Creative",    {"temperature":0.9, "top_p":0.95}),
    ]
    for name, cfg in configs:
        out = llm.generate_response(prompt, **cfg, max_tokens=150)
        print(f"\n{name} → {out[:200]}…")

compare_profiles(llm, "What are the benefits and challenges of space exploration?")



🔒 Conservative → <think>
Okay, so I need to figure out the benefits and challenges of space exploration. Let me start by brainstorming what I know. 

First, benefits. Well, I remember hearing that space exploration le…

⚖️ Balanced → <think>
Okay, so I need to figure out the benefits and challenges of space exploration. Let me start by brainstorming what I know. 

First, benefits. I remember hearing that space exploration leads to…

🎨 Creative → <think>
</think>

Space exploration has been a cornerstone of human innovation and curiosity for decades. The benefits and challenges of venturing into space are numerous and multifaceted. Here's a co…


## 🛠️ 9. Practical Use Cases

Example configurations for real-world tasks:

- **Factual QA**: low temp, low top_p  
- **Creative Writing**: high temp, high top_p  
- **Education**: moderate settings  
- **Brainstorming**: very high creativity  


In [ ]:
use_cases = [
    ("Factual QA",       "What is the capital of Canada?",             {"temperature":0.0, "top_p":0.1}),
    ("Creative Story",   "Start a mystery story set in London.",       {"temperature":0.9, "top_p":0.95}),
    ("Student Explainer","Explain relativity to a high schooler.",    {"temperature":0.3, "top_p":0.7}),
    ("Idea Generation",  "Five app ideas for eco-friendly living.",   {"temperature":0.9, "top_p":0.9}),
]
for name, prompt, cfg in use_cases:
    out = llm.generate_response(prompt, **cfg, max_tokens=100)
    print(f"\n— {name} ({cfg}) —\n{out}\n")



— Factual QA ({'temperature': 0.0, 'top_p': 0.1}) —
<think>
Okay, so the user is asking for the capital of Canada. Let me think. I remember that Canada is a country in North America, and I know some of its major cities like Toronto, Vancouver, and Montreal. But wait, the capital isn't always the biggest city. For example, in the US, the capital is Washington D.C., not New York.

I think the capital of Canada is Ottawa. But why do I remember that? Maybe from school or some trivia.


— Creative Story ({'temperature': 0.9, 'top_p': 0.95}) —
<think>
Okay, the user wants me to start a mystery story set in London. Let me think about how to approach this. First, I need to establish the setting. London is a great city for mysteries—lots of fog, history, hidden corners. Maybe set it in a specific part of London. The East End? Or maybe near the Thames. The time period? Maybe modern day but with a classic feel. 

The main character should be someone who can naturally get involved in a


— Studen

## 🎯 10. Interactive Playground

Customize your own prompt and parameters below!


In [ ]:
# Example interactive call
my_prompt = "Predict future trends in renewable energy."
my_temp   = 0.6
my_top_p  = 0.8
my_max    = 200

print("Your settings:", my_temp, my_top_p, my_max)
print(llm.generate_response(my_prompt,
                            temperature=my_temp,
                            top_p=my_top_p,
                            max_tokens=my_max))


Your settings: 0.6 0.8 200
<think>
Okay, so I need to predict future trends in renewable energy. Let me start by thinking about what I know. Renewable energy includes things like solar, wind, hydro, geothermal, and maybe bioenergy. The user probably wants a list of trends with explanations. Let me brainstorm some ideas.

First, solar and wind are getting cheaper. I remember reading that the cost of solar panels has dropped a lot over the years. So maybe that trend will continue. Also, technology improvements might make them more efficient. Like better solar cells or bigger wind turbines. Offshore wind is a thing too, right? Maybe that will expand.

Energy storage is a big deal because renewables can be intermittent. So battery technology, like lithium-ion, is important. But there's also other storage methods like pumped hydro or maybe new ones like solid-state batteries or flow batteries. If storage gets better and cheaper, that would help renewables a lot.

Electric vehicles are part 

## 📋 Summary & Best Practices

- **Temperature**:  
  - 0.0–0.3: factual & deterministic  
  - 0.4–0.7: balanced explanations  
  - 0.8–1.0: creative outputs  
- **Top-P**:  
  - 0.1–0.3: focused  
  - 0.4–0.8: moderate diversity  
  - 0.9–1.0: wide exploration  
- **Max Tokens**: tailor length to your use case  

> **Pro Tip:** Start conservative, then increase creativity until you hit your sweet spot!


## 📝 Hands-On Exercises: Solidify Your Understanding

Try these activities in your notebook to see parameter effects first-hand. For each exercise, pick a clear, concise prompt and record your observations.

---

### 1. Temperature Sweep  
- **Prompt:** “Write a short poem about spring.”  
- **Task:** Generate outputs at `temperature` = 0.0, 0.3, 0.6, 0.9.  
- **Observe:**  
  - How does word choice and structure change?  
  - Which temperature feels most “poetic” vs. “formulaic”?  

---

### 2. Top-P vs. Top-K Comparison  
- **Prompt:** “List five innovative uses of solar power.”  
- **Task:**  
  - Fix `temperature=0.7`.  
  - Generate with `top_p` = 0.3, 0.7, 0.95.  
  - Then fix `top_p=0.7` and try `top_k` = 5, 20, 100 (if supported).  
- **Observe:**  
  - Does top-p or top-k more strongly affect idea diversity?  
  - Are any outputs off-topic or repetitive?  

---

### 3. Max-Tokens Length Test  
- **Prompt:** “Explain quantum computing to a child.”  
- **Task:** Run with `max_tokens` = 50, 150, 300.  
- **Observe:**  
  - How does explanatory depth scale?  
  - Does the answer stay coherent when cut off?  

---

### 4. Profile Challenge: Conservative vs. Creative  
- **Prompt:** “Brainstorm a startup idea in the AI space.”  
- **Task:** Compare two configurations:  
  1. **Conservative**: `temperature=0.0`, `top_p=0.1`  
  2. **Creative**:   `temperature=0.9`, `top_p=0.9`  
- **Observe:**  
  - Which config yields more varied, unexpected ideas?  
  - Which is more grounded and practical?  

---

### 5. Task-Specific Tuning  
- **Summarization:** Prompt with a 200-word paragraph and try `temperature=0.0` vs. `0.5`.  
- **Translation:** Provide a sentence in English, request French translation at `temperature=0.0` vs. `0.8`.  
- **Classification:** Zero-shot sentiment analysis prompt, compare `max_tokens=3` vs. `max_tokens=10`.  

---

### 6. Quantitative Comparison  
- **Pick a fixed prompt** (e.g., “Describe the lifecycle of a star.”)  
- **Generate** 5 outputs each for two different settings.  
- **Compute:**  
  - **Average length** (# words)  
  - **Lexical diversity** (unique words ÷ total words)  
- **Discuss:** Which parameters maximize diversity? Which maximize conciseness?  

---

### 7. Creative Remix  
- **Prompt:** “Write the opening scene of a sci-fi adventure.”  
- **Task:** Tweak one parameter at a time to shift tone:  
  - Vary **temperature** only  
  - Vary **top_p** only  
  - Vary **max_tokens** only  
- **Outcome:** Create a short table summarizing how each tweak affected tone, detail, and length.

---

🧑‍💻 **Deliverable:** For each exercise, document your prompts, parameter settings, sample outputs, and a 1–2 sentence takeaway on how that parameter influenced the result. Happy experimenting!  
